# Lesson 04 - စက်ပစ္စည်းအသုံးပြုမှု ဒီဇိုင်န်းနမူနာ

ဤသင်ခန်းစာတွင် Microsoft Agent Framework (Python) ကိုအသုံးပြု၍ AI အေးဂျင့်များအတွက် **စက်ပစ္စည်းအသုံးပြုမှု** ဒီဇိုင်န်းနမူနာကို သင်ယူမည်ဖြစ်သည်။ ကျွန်ုပ်တို့သည် အောက်ပါအကြောင်းအရာများကို ဖော်ပြပါသည်-

- `@tool` decorator နှင့် typed parameters များဖြင့် ဖန်တီးထားသော function tools များကို တိတိကျကျ သတ်မှတ်ခြင်း
- မော်ဒယ်သည် စက်ပစ္စည်းတစ်ခုစီသည် ဘာလုပ်ဆောင်သည်ကို သိရှိနိုင်ရန် tool schemas များပေးသည့်နည်းလမ်း
- `approval_mode` ဖြင့် စက်ပစ္စည်းများ၏ ဆောင်ရွက်မှုကို ထိန်းချုပ်ခြင်း
- Pydantic models နှင့် `response_format` မှတဆင့် **ဖွဲ့စည်းအုပ်စုထားသော output** ရရှိစေရန်

ဤအခြေအနေမှာ ခရီးစဉ်စာရင်းသွင်းရေးအေးဂျင့်တစ်ဦးဖြစ်ပြီး၊ သွားရောက်နိုင်သောနေရာများကို ရှာဖွေပြသခြင်း၊ ရရှိနိုင်မှုကိုစစ်ဆေးခြင်းနှင့် လေကြောင်းသတင်းအချက်အလက်များအား ရယူနိုင်သည်။


## စတင်ပြင်ဆင်မှု


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool အမည်တွဲဖြင့် ကိရိယာများ သတ်မှတ်ခြင်း

`@tool` အမည်တွဲသည် ပုံမှန် Python function တစ်ခုအား agent ကခေါ်ဆိုနိုင်သော ကိရိယာတစ်ခုအဖြစ် ပြောင်းလဲပေးသည်။
အဓိကအချက်များ -

- **docstring** သည် စက်မော်ဒယ်မြင်နိုင်သည့် ကိရိယာ၏ ဖော်ပြချက် ဖြစ်သည်။
- **Type annotations** (ဖော်ပြချက်များပါဝင်သော `Annotated` အပါအဝင်) များက ကိရိယာ schema ကို သတ်မှတ်ပေးသည်။
- `approval_mode` သည် အသုံးပြုသူသည် တစ်ခါခေါ်တိုင်း အတည်ပြုရမည်ဟု ထိန်းချုပ်သည်။


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## အသုံးပြုသူမေးခွန်းကိုဖြေဖို့ မော်ဒယ်ကလိုအပ်တဲ့ သုံးခုလုံးသော ကိရိယာများအားလုံးကို client ထံပို့ပါ။


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ကိရိယာများဖြင့် စနစ်တကျ ထုတ်လွှင့်မှု

`response_format` ကို Pydantic မော်ဒယ်တစ်ခုအဖြစ် သတ်မှတ်ခြင်းအားဖြင့်၊ အေးဂျင့်သည် အခမဲ့ စာသားမျိုးမဟုတ်ဘဲ ကောင်းမွန်စွာ အမျိုးအစား သတ်မှတ်ထားသော JSON ဂဏန်းတစ်ခုကို ပြန်လည်ထုတ်ပေးရန် မဖြစ်မနေ ထိန်းမှာပေးသည်။ ၎င်းသည် နောက်ဆက်တွဲ ကုဒ်များမှ အဆိုပါရလဒ်ကို ပရိုဂရမ်ရေးမတိုင်ခင် အသုံးပြုသင့်သော အချိန်တွင် အသုံးဝင်သည်။


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ကိရိယာအတည်ပြုမှု ပုံစံများ

`@tool` တွင်ရှိသော `approval_mode` ပါရာမီတာသည် ကိရိယာခေါ်ဆိုမှုများကို ဖော်ဆောင်ခြင်းမပြုလုပ်ခင် လူ့အတည်ပြုမှု လိုအပ်မလိုကို ထိန်းချုပ်ပေးသည်-

| Mode | အပြုအမူ |
|---|---|
| `"never_require"` | ကိရိယာကို အလိုအလျောက် သွားလုပ်ဆောင်သည် — အသုံးပြုသူ၏ အတည်ပြုချက် မလိုအပ်ပါ။ |
| `"always_require"` | ခေါ်ဆိုမှုတိုင်းကို ဖော်ဆောင်ခြင်းမပြုမီ အသုံးပြုသူမှ အတည်ပြုရမည်။ |

ဘက်ဆက်ရလဒ်များ ရှိသော ကိရိယာများ (ဥပမာ- သွားကြေးမှာယူခြင်း၊ ခရက်ဒစ်ကတ်အား သင့်လျော်မှုစစ်ဆေးခြင်း) အတွက် `"always_require"` ကို အသုံးပြုပါ၊ ဤကဲ့သို့ လူတစ်ယောက် အလျှောက်ကိရိယာအတွင်းတွင် ရှိနေစေရန်။


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## အကျဉ်းချုပ်

ဤသင်ခန်းစာတွင် သင်သည် အောက်ပါအတိုင်း လေ့လာသင့်သည်-

1. `@tool` decorator ကို အသုံးပြု၍ ပရမိတာများနှင့် ကိုယ်ရေးအချက်အလက်စာသားများပါသော ကိရိယာ schema အဖြစ် **ကိရိယာများကို သတ်မှတ်ခြင်း**။
2. ကိုယ်စားလှယ်သည် ရှုပ်ထွေးသော စုံစမ်းမေးမြန်းမှုများကို ဖြေရှင်းနိုင်ရန် **ကိရိယာများစွာကို တတိယဆုံး ကျွန်တော်အသုံးပြုခြင်း**။
3. Pydantic မော်ဒယ်ကို `response_format` အဖြစ် ပေးပို့၍ **ဖွဲ့စည်းထားသော output ကို ပြန်ပေးပို့ခြင်း**။
4. အရေးပါတဲ့ လုပ်ငန်းစဉ်များအတွက် လူသား ပါဝင်စောင့်ကြည့်မှုအတွက် `approval_mode` ဖြင့် **ကိရိယာခွင့်ပြုချက် ထိန်းချုပ်ခြင်း**။

ဤပုံစံများသည် အပြင်ပေါ် စနစ်များနှင့် အဆင်ပြေစွာ ဆက်သွယ်နိုင်ပြီး ယုံကြည်စိတ်ချရသော ထုတ်လုပ်မှုအဆင့် ကိုယ်စားလှယ်များ တည်ဆောက်ရန် အခြေခံကို ဖွဲ့စည်းပေးသည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ဂရုပြုချက်**၊  
ဤစာရွက်ကို AI ဘာသာပြန်ဝန်ဆောင်မှုဖြစ်သည့် [Co-op Translator](https://github.com/Azure/co-op-translator) ကို အသုံးပြု၍ ဘာသာပြန်ထားခြင်းဖြစ်ပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှုအတွက် ကြိုးပမ်းပါသော်လည်း အလိုအလျောက်ဘာသာပြန်ခြင်း၌ မှားယွင်းမှုများ သို့မဟုတ် မ မှန်ကန်မှုများ ရှိနိုင်ပါသည်။ မူရင်းစာရွက်ကို တိုင်းရင်းဘာသာဖြင့် စံချိန်စံညွှန်းအဖြစ် သတ်မှတ်သင့်ပါသည်။ အရေးကြီးသော အချက်အလက်များအတွက် လူ့ဘာသာပြန်မှု အသုံးပြုရန် တိုက်တွန်းပါသည်။ ဤဘာသာပြန်ချက် အသုံးပြုမှုကြောင့် ဖြစ်ပေါ်နိုင်သည့် နားမလည်မှုများ သို့မဟုတ် မှားယွင်းဖတ်ရှုမှုများအတွက် ကျွန်ုပ်တို့ မည်သည့် တာဝန်ကိုမှ မယူဆောင်ပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
